# OSRS Skilling Success Formula

Source: https://oldschool.runescape.wiki/w/Template:Skilling_success_chart

## Formula

$$f(L) = \frac{\lfloor C_{Low} \cdot \frac{99-L}{98} + C_{High} \cdot \frac{L-1}{98} + 0.5 \rfloor + 1}{256}$$

| Variable | Meaning |
|---|---|
| `L` | Player's skill level |
| `C_Low` | Success numerator at level 1 (from RuneScript, used over /256) |
| `C_High` | Success numerator at level 99 (from RuneScript, used over /256) |

> **Note:** C_Low and C_High are defined in RuneScript as fractions of 255, but the engine divides by 256 — creating a slight discrepancy vs. community-published percentages.

## Proof: Yew Tree with Dragon Axe

From wiki raw source:
- `C_Low = 15`, `C_High = 47`, requirement = 61 Woodcutting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

def success_chance(level, c_low, c_high, crystal_axe=False, barbarian_gathering=False):
    """OSRS skilling success chance formula.

    crystal_axe:          if True, c_low/c_high are treated as dragon axe values and
                          scaled by (1.15/1.10) to reflect crystal axe's 15% efficiency
                          boost over rune axe vs dragon axe's 10% boost.

    barbarian_gathering:  if True, applies a 50% second-chance on failure:
                          effective = p + (1 - p) * 0.5  =  0.5 + 0.5 * p
    """
    if crystal_axe:
        c_low  = c_low  * (1.15 / 1.10)
        c_high = c_high * (1.15 / 1.10)

    numerator = math.floor(c_low * (99 - level) / 98 + c_high * (level - 1) / 98 + 0.5) + 1
    p = numerator / 256
    p = max(0.0, min(1.0, p))

    if barbarian_gathering:
        p = p + (1 - p) * 0.5

    return p


---
## Mining

Mining uses the same formula but **C_Low/C_High do not vary by pickaxe** — all pickaxes roll against the same per-roll odds for a given rock. Pickaxe tier only affects roll frequency (ticks per swing).

Sandstone is an exception: each ore size has its own C values, rolled independently on each tick.

> Amethyst, Granite, and Lovakite are tagged "Needs skilling success chart" on the wiki — no data available yet.

In [ ]:
# C_Low / C_High values sourced from OSRS wiki raw wikitext (?action=raw)
# XP values sourced from https://oldschool.runescape.wiki/w/Woodcutting
# Trees without crystal axe wiki data use crystal_axe=True estimation in success_chance()
TREES = {
    "Tree": {
        "level_req": 1, "xp": 25,
        "axes": {
            "bronze":  {"c_low": 64,  "c_high": 200, "req": 1},
            "iron":    {"c_low": 96,  "c_high": 300, "req": 1},
            "steel":   {"c_low": 128, "c_high": 400, "req": 6},
            "black":   {"c_low": 144, "c_high": 450, "req": 11},
            "mithril": {"c_low": 160, "c_high": 500, "req": 21},
            "adamant": {"c_low": 192, "c_high": 600, "req": 31},
            "rune":    {"c_low": 224, "c_high": 700, "req": 41},
            "dragon":  {"c_low": 240, "c_high": 750, "req": 61},
            "crystal": {"c_low": 250, "c_high": 800, "req": 71},  # wiki exact
        },
    },
    "Oak": {
        "level_req": 15, "xp": 37.5,
        "axes": {
            "bronze":  {"c_low": 32,  "c_high": 100, "req": 15},
            "iron":    {"c_low": 48,  "c_high": 150, "req": 15},
            "steel":   {"c_low": 64,  "c_high": 200, "req": 15},
            "black":   {"c_low": 72,  "c_high": 225, "req": 15},
            "mithril": {"c_low": 80,  "c_high": 250, "req": 21},
            "adamant": {"c_low": 96,  "c_high": 300, "req": 31},
            "rune":    {"c_low": 112, "c_high": 350, "req": 41},
            "dragon":  {"c_low": 120, "c_high": 375, "req": 61},
        },
    },
    "Willow": {
        "level_req": 30, "xp": 67.5,
        "axes": {
            "bronze":  {"c_low": 16, "c_high": 50,  "req": 30},
            "iron":    {"c_low": 24, "c_high": 75,  "req": 30},
            "steel":   {"c_low": 32, "c_high": 100, "req": 30},
            "black":   {"c_low": 36, "c_high": 112, "req": 30},
            "mithril": {"c_low": 40, "c_high": 125, "req": 30},
            "adamant": {"c_low": 48, "c_high": 150, "req": 31},
            "rune":    {"c_low": 56, "c_high": 175, "req": 41},
            "dragon":  {"c_low": 60, "c_high": 187, "req": 61},
        },
    },
    "Teak": {
        "level_req": 35, "xp": 85,
        "axes": {
            "bronze":  {"c_low": 15, "c_high": 46,  "req": 35},
            "iron":    {"c_low": 23, "c_high": 70,  "req": 35},
            "steel":   {"c_low": 31, "c_high": 93,  "req": 35},
            "black":   {"c_low": 35, "c_high": 102, "req": 35},
            "mithril": {"c_low": 39, "c_high": 117, "req": 35},
            "adamant": {"c_low": 47, "c_high": 140, "req": 35},
            "rune":    {"c_low": 55, "c_high": 164, "req": 41},
            "dragon":  {"c_low": 60, "c_high": 190, "req": 61},
            "crystal": {"c_low": 60, "c_high": 200, "req": 71},  # wiki exact
        },
    },
    "Maple": {
        "level_req": 45, "xp": 100,
        "axes": {
            "bronze":  {"c_low": 8,  "c_high": 25, "req": 45},
            "iron":    {"c_low": 12, "c_high": 37, "req": 45},
            "steel":   {"c_low": 16, "c_high": 50, "req": 45},
            "black":   {"c_low": 18, "c_high": 56, "req": 45},
            "mithril": {"c_low": 20, "c_high": 62, "req": 45},
            "adamant": {"c_low": 24, "c_high": 75, "req": 45},
            "rune":    {"c_low": 28, "c_high": 87, "req": 45},
            "dragon":  {"c_low": 30, "c_high": 93, "req": 61},
        },
    },
    "Mahogany": {
        "level_req": 50, "xp": 125,
        "axes": {
            "bronze":  {"c_low": 8,  "c_high": 25, "req": 50},
            "iron":    {"c_low": 12, "c_high": 38, "req": 50},
            "steel":   {"c_low": 16, "c_high": 50, "req": 50},
            "black":   {"c_low": 18, "c_high": 54, "req": 50},
            "mithril": {"c_low": 20, "c_high": 63, "req": 50},
            "adamant": {"c_low": 25, "c_high": 75, "req": 50},
            "rune":    {"c_low": 29, "c_high": 88, "req": 50},
            "dragon":  {"c_low": 34, "c_high": 94, "req": 61},
            "crystal": {"c_low": 36, "c_high": 97, "req": 71},  # wiki exact
        },
    },
    "Arctic Pine": {
        "level_req": 54, "xp": 40,
        "axes": {
            "bronze":  {"c_low": 6,  "c_high": 30,  "req": 54},
            "iron":    {"c_low": 8,  "c_high": 44,  "req": 54},
            "steel":   {"c_low": 11, "c_high": 60,  "req": 54},
            "black":   {"c_low": 13, "c_high": 67,  "req": 54},
            "mithril": {"c_low": 14, "c_high": 74,  "req": 54},
            "adamant": {"c_low": 17, "c_high": 90,  "req": 54},
            "rune":    {"c_low": 20, "c_high": 104, "req": 54},
            "dragon":  {"c_low": 21, "c_high": 112, "req": 61},
        },
    },
    "Yew": {
        "level_req": 60, "xp": 175,
        "axes": {
            "bronze":  {"c_low": 4,  "c_high": 12, "req": 60},
            "iron":    {"c_low": 6,  "c_high": 19, "req": 60},
            "steel":   {"c_low": 8,  "c_high": 25, "req": 60},
            "black":   {"c_low": 9,  "c_high": 28, "req": 60},
            "mithril": {"c_low": 10, "c_high": 31, "req": 60},
            "adamant": {"c_low": 12, "c_high": 37, "req": 60},
            "rune":    {"c_low": 14, "c_high": 44, "req": 60},
            "dragon":  {"c_low": 15, "c_high": 47, "req": 61},
        },
    },
    "Magic": {
        "level_req": 75, "xp": 250,
        "axes": {
            "bronze":  {"c_low": 2, "c_high": 6,  "req": 75},
            "iron":    {"c_low": 3, "c_high": 9,  "req": 75},
            "steel":   {"c_low": 4, "c_high": 12, "req": 75},
            "black":   {"c_low": 5, "c_high": 13, "req": 75},
            "mithril": {"c_low": 5, "c_high": 15, "req": 75},
            "adamant": {"c_low": 6, "c_high": 18, "req": 75},
            "rune":    {"c_low": 7, "c_high": 21, "req": 75},
            "dragon":  {"c_low": 7, "c_high": 22, "req": 75},
        },
    },
    "Redwood": {
        "level_req": 90, "xp": 380,
        "axes": {
            "bronze":  {"c_low": 2, "c_high": 6,  "req": 90},
            "iron":    {"c_low": 3, "c_high": 9,  "req": 90},
            "steel":   {"c_low": 4, "c_high": 12, "req": 90},
            "black":   {"c_low": 4, "c_high": 14, "req": 90},
            "mithril": {"c_low": 5, "c_high": 15, "req": 90},
            "adamant": {"c_low": 6, "c_high": 18, "req": 90},
            "rune":    {"c_low": 7, "c_high": 21, "req": 90},
            "dragon":  {"c_low": 7, "c_high": 30, "req": 90},
            "crystal": {"c_low": 8, "c_high": 35, "req": 90},  # wiki exact
        },
    },
}

def get_cut_chance(tree, axe, level, crystal_axe=False, barbarian_gathering=False):
    """Look up cut chance from TREES map.

    For 'crystal' axe on trees without wiki data, falls back to estimating
    from dragon axe values using the crystal_axe=True scaling (x1.15/1.10).
    """
    tree_data = TREES[tree]
    axes = tree_data["axes"]

    if axe == "crystal" and "crystal" not in axes:
        d = axes["dragon"]
        return success_chance(level, d["c_low"], d["c_high"],
                                          crystal_axe=True,
                                          barbarian_gathering=barbarian_gathering)

    a = axes[axe]
    if level < a["req"]:
        raise ValueError(f"{axe} axe requires level {a['req']} (you have {level})")
    if level < tree_data["level_req"]:
        raise ValueError(f"{tree} tree requires level {tree_data['level_req']} (you have {level})")
    return success_chance(level, a["c_low"], a["c_high"],
                                      crystal_axe=crystal_axe,
                                      barbarian_gathering=barbarian_gathering)

def xp_per_action_wc(tree, level, axe="dragon", crystal_axe=False, barbarian_gathering=False):
    """Expected XP per woodcutting action = success_chance * xp_per_log."""
    chance = get_cut_chance(tree, axe, level,
                            crystal_axe=crystal_axe,
                            barbarian_gathering=barbarian_gathering)
    return chance * TREES[tree]["xp"]

In [ ]:
# C_Low / C_High sourced from OSRS wiki raw wikitext (?action=raw)
# XP values sourced from https://oldschool.runescape.wiki/w/Mining
# Sandstone/Granite XP is per-size. Missing wiki data: Lovakite (65).
ROCKS = {
    "Copper":     {"c_low": 100, "c_high": 350, "level_req": 1,  "xp": 17.5},
    "Tin":        {"c_low": 100, "c_high": 350, "level_req": 1,  "xp": 17.5},
    "Iron":       {"c_low": 96,  "c_high": 350, "level_req": 15, "xp": 35},
    "Silver":     {"c_low": 25,  "c_high": 200, "level_req": 20, "xp": 40},
    "Coal":       {"c_low": 16,  "c_high": 100, "level_req": 30, "xp": 50},
    "Gold":       {"c_low": 7,   "c_high": 75,  "level_req": 40, "xp": 65},
    "Mithril":    {"c_low": 4,   "c_high": 50,  "level_req": 55, "xp": 80},
    "Adamantite": {"c_low": 2,   "c_high": 25,  "level_req": 70, "xp": 95},
    "Runite":     {"c_low": 1,   "c_high": 18,  "level_req": 85, "xp": 125},
    "Amethyst":   {"c_low": -18, "c_high": 10,  "level_req": 92, "xp": 240},
    # Gem rocks — charged glory amulet triples C values (Mod Ash; wiki confirmed)
    # Source: https://oldschool.runescape.wiki/w/Gem_rocks
    "Gem rock":         {"c_low": 28, "c_high": 70,  "level_req": 40, "xp": 65},
    "Gem rock (glory)": {"c_low": 84, "c_high": 210, "level_req": 40, "xp": 65},
    # Sandstone: each size is a separate roll; xp varies per size
    "Sandstone_1kg":  {"c_low": 25, "c_high": 200, "level_req": 35, "xp": 30},
    "Sandstone_2kg":  {"c_low": 16, "c_high": 100, "level_req": 35, "xp": 40},
    "Sandstone_5kg":  {"c_low": 8,  "c_high": 75,  "level_req": 35, "xp": 50},
    "Sandstone_10kg": {"c_low": 4,  "c_high": 50,  "level_req": 35, "xp": 60},
    # Granite: each size is a separate roll; xp varies per size
    "Granite_500g": {"c_low": 16, "c_high": 100, "level_req": 45, "xp": 50},
    "Granite_2kg":  {"c_low": 8,  "c_high": 75,  "level_req": 45, "xp": 60},
    "Granite_5kg":  {"c_low": 6,  "c_high": 64,  "level_req": 45, "xp": 75},
}

# Priority order: largest size first (first success wins)
SANDSTONE_SIZES = ["Sandstone_10kg", "Sandstone_5kg", "Sandstone_2kg", "Sandstone_1kg"]
GRANITE_SIZES   = ["Granite_5kg", "Granite_2kg", "Granite_500g"]

def get_mine_chance(rock, level, barbarian_gathering=False):
    """Return per-roll success chance for mining a given rock at a given level.
    Result is clamped to [0, 1] before barbarian_gathering is applied.
    """
    r = ROCKS[rock]
    if level < r["level_req"]:
        raise ValueError(f"{rock} requires level {r['level_req']} (you have {level})")
    return success_chance(level, r["c_low"], r["c_high"],
                                      barbarian_gathering=barbarian_gathering)

def xp_per_action_mining(rock, level, barbarian_gathering=False):
    """Expected XP per mining action.

    For plain rocks: success_chance * xp.  BG is applied at the roll level
    (flat 50% second chance on failure), which is correct for solo resources.

    For Sandstone / Granite: sizes are rolled largest-first; the first success
    wins.  BG is applied at the ACTION level — it fires once if the entire
    priority chain fails, then re-runs the chain.  This avoids giving each
    individual size its own independent BG second-chance (which would
    effectively grant multiple second-chances per tick).

        E[XP | BG, shared] = E[XP | no BG] * (1 + 0.5 * P(all sizes fail))
    """
    if rock in ("Sandstone", "Granite"):
        sizes = SANDSTONE_SIZES if rock == "Sandstone" else GRANITE_SIZES
        level_req = ROCKS[sizes[0]]["level_req"]
        if level < level_req:
            raise ValueError(f"{rock} requires level {level_req} (you have {level})")

        expected_xp    = 0.0
        prob_no_larger = 1.0
        for size in sizes:
            p = get_mine_chance(size, level, barbarian_gathering=False)
            expected_xp    += prob_no_larger * p * ROCKS[size]["xp"]
            prob_no_larger *= (1 - p)
        # prob_no_larger is now P(all sizes failed)
        if barbarian_gathering:
            expected_xp *= (1 + 0.5 * prob_no_larger)
        return expected_xp

    chance = get_mine_chance(rock, level, barbarian_gathering=barbarian_gathering)
    return chance * ROCKS[rock]["xp"]


---
## Fishing

Most fish have a single `c_low`/`c_high` pair. Harponable fish (Tuna, Swordfish, Shark) have three sets under `"harpoons"`: `regular`, `dragon`, `crystal`.

Anglerfish has two bait variants (`sandworms`, `diabolic_worms`). Dark crab has a bonus variant for Elite Wilderness Diary completion.

Note: several fish share a fishing spot and are caught simultaneously — shrimp/anchovies, sardine/herring, trout/salmon, tuna/swordfish.

In [ ]:
# C_Low / C_High sourced from OSRS wiki raw wikitext (?action=raw) on Raw_* item pages
# XP sourced from https://oldschool.runescape.wiki/w/Fishing
# Harponable fish have "harpoons": {"regular", "dragon", "crystal"}
FISH = {
    # --- Small net ---
    "Shrimp": {
        "level_req": 1, "xp": 10, "method": "small net",
        "c_low": 48, "c_high": 256,
    },
    "Anchovies": {
        "level_req": 15, "xp": 40, "method": "small net",
        "c_low": 24, "c_high": 128,
    },
    # --- Bait ---
    "Sardine": {
        "level_req": 5, "xp": 20, "method": "bait",
        "c_low": 32, "c_high": 192,
    },
    "Herring": {
        "level_req": 10, "xp": 30, "method": "bait",
        "c_low": 24, "c_high": 128,
    },
    "Pike": {
        "level_req": 25, "xp": 60, "method": "bait",
        "c_low": 16, "c_high": 96,
    },
    # --- Fly fishing ---
    "Trout": {
        "level_req": 20, "xp": 50, "method": "fly fishing",
        "c_low": 32, "c_high": 192,
    },
    "Salmon": {
        "level_req": 30, "xp": 70, "method": "fly fishing",
        "c_low": 16, "c_high": 96,
    },
    # --- Harpoon (3 variants) ---
    "Tuna": {
        "level_req": 35, "xp": 80, "method": "harpoon",
        "harpoons": {
            "regular": {"c_low": 8,  "c_high": 64},
            "dragon":  {"c_low": 9,  "c_high": 76},
            "crystal": {"c_low": 10, "c_high": 86},
        },
    },
    "Swordfish": {
        "level_req": 50, "xp": 100, "method": "harpoon",
        "harpoons": {
            "regular": {"c_low": 4, "c_high": 48},
            "dragon":  {"c_low": 4, "c_high": 57},
            "crystal": {"c_low": 5, "c_high": 64},
        },
    },
    "Shark": {
        "level_req": 76, "xp": 110, "method": "harpoon",
        "harpoons": {
            "regular": {"c_low": 3, "c_high": 40},
            "dragon":  {"c_low": 3, "c_high": 48},
            "crystal": {"c_low": 4, "c_high": 54},
        },
    },
    # --- Cage ---
    "Lobster": {
        "level_req": 40, "xp": 90, "method": "cage",
        "c_low": 6, "c_high": 95,
    },
    # --- Big net ---
    "Bass": {
        "level_req": 46, "xp": 100, "method": "big net",
        "c_low": 3, "c_high": 40,
    },
    # --- Small net (Piscatoris) ---
    "Monkfish": {
        "level_req": 62, "xp": 120, "method": "small net",
        "c_low": 48, "c_high": 90,
    },
    # --- Karambwan pot ---
    "Karambwan": {
        "level_req": 65, "xp": 50, "method": "karambwan vessel",
        "c_low": 5, "c_high": 160,
    },
    # --- Bait (two bait types) ---
    "Anglerfish": {
        "level_req": 82, "xp": 120, "method": "bait",
        "baits": {
            "sandworms":      {"c_low": 3,  "c_high": 36},
            "diabolic_worms": {"c_low": 13, "c_high": 100},
        },
    },
    # --- Cage (Wilderness) ---
    "Dark Crab": {
        "level_req": 85, "xp": 130, "method": "cage",
        "c_low": 3, "c_high": 40,
        "elite_diary": {"c_low": 13, "c_high": 55},
    },
    # --- Barbarian rod fishing (heavy rod + bait; shared spot, priority largest first) ---
    # Source: https://oldschool.runescape.wiki/w/Fishing_spot_(barbarian)
    "Leaping Trout": {
        "level_req": 48, "xp": 50, "method": "barbarian rod",
        "c_low": 32, "c_high": 192,
    },
    "Leaping Salmon": {
        "level_req": 58, "xp": 70, "method": "barbarian rod",
        "c_low": 16, "c_high": 96,
    },
    "Leaping Sturgeon": {
        "level_req": 70, "xp": 80, "method": "barbarian rod",
        "c_low": 8, "c_high": 64,
    },
}

# Shared fishing spots — priority order is highest-level fish first.
# On each action, rolls are made in priority order; the first success wins.
# Fish not yet unlocked at the given level are skipped.
FISHING_SPOTS = {
    "Shrimp":    ["Anchovies", "Shrimp"],
    "Anchovies": ["Anchovies", "Shrimp"],
    "Sardine":   ["Herring",   "Sardine"],
    "Herring":   ["Herring",   "Sardine"],
    "Trout":     ["Salmon",    "Trout"],
    "Salmon":    ["Salmon",    "Trout"],
    "Tuna":      ["Swordfish", "Tuna"],
    "Swordfish": ["Swordfish", "Tuna"],
    "Pike":       ["Pike"],
    "Lobster":    ["Lobster"],
    "Bass":       ["Bass"],
    "Monkfish":   ["Monkfish"],
    "Karambwan":  ["Karambwan"],
    "Shark":      ["Shark"],
    "Anglerfish": ["Anglerfish"],
    "Dark Crab":  ["Dark Crab"],
    # Barbarian fishing spot — all three share one spot, rolled largest first
    "Leaping Trout":    ["Leaping Sturgeon", "Leaping Salmon", "Leaping Trout"],
    "Leaping Salmon":   ["Leaping Sturgeon", "Leaping Salmon", "Leaping Trout"],
    "Leaping Sturgeon": ["Leaping Sturgeon", "Leaping Salmon", "Leaping Trout"],
}
# Ticks per resource roll, by fishing spot anchor
FISHING_SPOT_TICKS = {
    "Shrimp":        5,   # small net
    "Sardine":       5,   # bait
    "Pike":          5,   # bait
    "Trout":         5,   # fly fishing
    "Lobster":       6,   # cage
    "Tuna":          6,   # harpoon
    "Bass":          6,   # big net
    "Monkfish":      5,   # small net
    "Karambwan":     4,   # karambwan vessel
    "Anglerfish":    5,   # bait
    "Shark":         6,   # harpoon
    "Dark Crab":     6,   # cage (wilderness)
    "Leaping Trout": 5,   # barbarian rod
}

def get_fish_chance(fish, level, harpoon="regular", bait="sandworms",
                    elite_diary=False, barbarian_gathering=False):
    """Return per-roll fishing success chance for a single fish type."""
    f = FISH[fish]
    if level < f["level_req"]:
        raise ValueError(f"{fish} requires level {f['level_req']} (you have {level})")

    if "harpoons" in f:
        h = f["harpoons"][harpoon]
        return success_chance(level, h["c_low"], h["c_high"],
                                          barbarian_gathering=barbarian_gathering)
    elif "baits" in f:
        b = f["baits"][bait]
        return success_chance(level, b["c_low"], b["c_high"],
                                          barbarian_gathering=barbarian_gathering)
    elif elite_diary and "elite_diary" in f:
        d = f["elite_diary"]
        return success_chance(level, d["c_low"], d["c_high"],
                                          barbarian_gathering=barbarian_gathering)
    else:
        return success_chance(level, f["c_low"], f["c_high"],
                                          barbarian_gathering=barbarian_gathering)

def xp_per_action_fishing(fish, level, harpoon="regular", bait="sandworms",
                           elite_diary=False, barbarian_gathering=False):
    """Expected XP per fishing action.

    For solo spots: BG is applied at the roll level (flat 50% second chance),
    which is correct for a single-fish node.

    For shared spots (2+ reachable fish): BG is applied at the ACTION level —
    it fires once only if the entire priority chain fails, then re-runs the
    chain.  Applying BG per-fish would grant each fish its own independent
    second-chance, effectively giving more than one extra attempt per tick.

        E[XP | BG, shared] = E[XP | no BG] * (1 + 0.5 * P(all fish fail))
    """
    spot      = FISHING_SPOTS[fish]
    reachable = [f for f in spot if level >= FISH[f]["level_req"]]
    is_shared = len(reachable) > 1

    expected_xp    = 0.0
    prob_no_higher = 1.0

    for f in reachable:
        # For shared spots with BG enabled, compute base chain without BG;
        # the action-level multiplier is applied afterwards.
        per_fish_bg = barbarian_gathering and not is_shared
        p = get_fish_chance(f, level, harpoon=harpoon, bait=bait,
                            elite_diary=elite_diary,
                            barbarian_gathering=per_fish_bg)
        expected_xp    += prob_no_higher * p * FISH[f]["xp"]
        prob_no_higher *= (1 - p)

    if barbarian_gathering and is_shared:
        # prob_no_higher is now P(all fish in chain failed)
        expected_xp *= (1 + 0.5 * prob_no_higher)

    return expected_xp


In [ ]:
# ── Woodcutting: XP per action, dragon axe, key trees, with/without Barbarian Gathering ──
trees_to_plot = ["Willow", "Maple", "Yew", "Magic", "Redwood"]
colors = plt.cm.tab10(np.linspace(0, 0.9, len(trees_to_plot)))

fig, ax = plt.subplots(figsize=(12, 6))

for tree, color in zip(trees_to_plot, colors):
    axe_req   = TREES[tree]["axes"]["dragon"]["req"]
    tree_req  = TREES[tree]["level_req"]
    start     = max(tree_req, axe_req)
    levels    = list(range(start, 100))

    xp_norm = [xp_per_action_wc(tree, l, axe="dragon") for l in levels]
    xp_bg   = [xp_per_action_wc(tree, l, axe="dragon", barbarian_gathering=True) for l in levels]

    ax.plot(levels, xp_norm, color=color, linewidth=2,   label=tree)
    ax.plot(levels, xp_bg,   color=color, linewidth=1.5, linestyle="--", alpha=0.7)

# Legend: tree colours + line style key
handles, labels = ax.get_legend_handles_labels()
from matplotlib.lines import Line2D
handles += [Line2D([0], [0], color="black", lw=2,   label="Normal"),
            Line2D([0], [0], color="black", lw=1.5, linestyle="--", alpha=0.7, label="+ Barbarian Gathering")]
ax.legend(handles=handles, ncol=2)

ax.set_title("Woodcutting — Expected XP per Action (Dragon Axe)")
ax.set_xlabel("Woodcutting Level")
ax.set_ylabel("Expected XP per Action")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("wc_xp_per_action.png", dpi=150)
plt.show()

In [ ]:
# ── Mining: XP per action, key rocks, with/without Barbarian Gathering ──
rocks_to_plot = ["Iron", "Coal", "Gold", "Mithril", "Adamantite", "Runite", "Amethyst"]
colors = plt.cm.tab10(np.linspace(0, 0.9, len(rocks_to_plot)))

fig, ax = plt.subplots(figsize=(12, 6))

for rock, color in zip(rocks_to_plot, colors):
    start  = ROCKS[rock]["level_req"]
    levels = list(range(start, 100))

    xp_norm = [xp_per_action_mining(rock, l) for l in levels]
    xp_bg   = [xp_per_action_mining(rock, l, barbarian_gathering=True) for l in levels]

    ax.plot(levels, xp_norm, color=color, linewidth=2,   label=rock)
    ax.plot(levels, xp_bg,   color=color, linewidth=1.5, linestyle="--", alpha=0.7)

handles, labels = ax.get_legend_handles_labels()
handles += [Line2D([0], [0], color="black", lw=2,   label="Normal"),
            Line2D([0], [0], color="black", lw=1.5, linestyle="--", alpha=0.7, label="+ Barbarian Gathering")]
ax.legend(handles=handles, ncol=2)

ax.set_title("Mining — Expected XP per Action")
ax.set_xlabel("Mining Level")
ax.set_ylabel("Expected XP per Action")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("mining_xp_per_action.png", dpi=150)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.transforms as mtransforms

# ── Endless Harvest settings ──
XP_MULTIPLIER  = 16
MINUTES        = 30
TICKS          = int(MINUTES * 60 / 0.6)   # 3000 ticks

WC_TICKS      = 4
MINING_TICKS  = 3
LEVEL         = 99

def get_fish_tick(fish: str) -> int:
    return FISHING_SPOT_TICKS[fish]


rows = []

for tree, data in TREES.items():
    rolls   = TICKS / WC_TICKS
    xpa     = xp_per_action_wc(tree, LEVEL, axe="rune")
    xpa_x2  = xpa * 2
    rows.append({"Skill": "Woodcutting", "Resource": tree,
                 "Ticks/Roll": WC_TICKS, "Rolls/30min": int(rolls),
                 "XP/Action": round(xpa, 2), "XP/Action (×2)": round(xpa_x2, 2),
                 "XP in 30min": round(rolls * xpa_x2 * XP_MULTIPLIER)})

for rock in ["Copper", "Tin", "Iron", "Silver", "Coal", "Gold",
             "Mithril", "Adamantite", "Runite", "Amethyst",
             "Gem rock", "Gem rock (glory)"]:
    rolls = TICKS / MINING_TICKS
    xpa   = xp_per_action_mining(rock, LEVEL)
    xpa_x2 = xpa * 2
    rows.append({"Skill": "Mining", "Resource": rock,
                 "Ticks/Roll": MINING_TICKS, "Rolls/30min": int(rolls),
                 "XP/Action": round(xpa, 2), "XP/Action (×2)": round(xpa_x2, 2),
                 "XP in 30min": round(rolls * xpa_x2 * XP_MULTIPLIER)})

for rock in ["Sandstone", "Granite"]:
    rolls = TICKS / MINING_TICKS
    xpa   = xp_per_action_mining(rock, LEVEL)
    xpa_x2 = xpa * 2
    rows.append({"Skill": "Mining", "Resource": rock + " (priority roll)",
                 "Ticks/Roll": MINING_TICKS, "Rolls/30min": int(rolls),
                 "XP/Action": round(xpa, 2), "XP/Action (×2)": round(xpa_x2, 2),
                 "XP in 30min": round(rolls * xpa_x2 * XP_MULTIPLIER)})

fishing_entries = [
    ("Shrimp/Anchovies",              "Shrimp",        dict()),
    ("Sardine/Herring",               "Sardine",       dict()),
    ("Pike",                          "Pike",          dict()),
    ("Trout/Salmon",                  "Trout",         dict()),
    ("Lobster",                       "Lobster",       dict()),
    ("Tuna/Swordfish",                "Tuna",          dict(harpoon="regular")),
    ("Bass",                          "Bass",          dict()),
    ("Monkfish",                      "Monkfish",      dict()),
    ("Karambwan",                     "Karambwan",     dict()),
    ("Shark",                         "Shark",         dict(harpoon="regular")),
    ("Anglerfish",                    "Anglerfish",    dict(bait="sandworms")),
    ("Dark Crab",                     "Dark Crab",     dict()),
    ("Leaping Trout/Salmon/Sturgeon", "Leaping Trout", dict()),
]
for label, fish, kwargs in fishing_entries:
    rolls  = TICKS / get_fish_tick(fish)
    xpa    = xp_per_action_fishing(fish, LEVEL, **kwargs)
    xpa_x2 = xpa * 2
    rows.append({"Skill": "Fishing", "Resource": label,
                 "Ticks/Roll":  get_fish_tick(fish), "Rolls/30min": int(rolls),
                 "XP/Action": round(xpa, 2), "XP/Action (×2)": round(xpa_x2, 2),
                 "XP in 30min": round(rolls * xpa_x2 * XP_MULTIPLIER)})

df = pd.DataFrame(rows)
df = df.sort_values(["Skill", "XP in 30min"], ascending=[True, False]).reset_index(drop=True)
df["XP in 30min"] = df["XP in 30min"].map("{:,.0f}".format)

SKILL_COLORS = {"Woodcutting": "#d4f0c0", "Mining": "#d0e8f5", "Fishing": "#fde9c0"}
ROW_ALT      = {"Woodcutting": "#eaf7e0", "Mining": "#e5f2fa", "Fishing": "#fef4db"}

col_labels = list(df.columns)
n_rows, n_cols = df.shape

fig, ax = plt.subplots(figsize=(14, 0.45 * n_rows + 1.2))
ax.axis("off")
ax.set_position([0, 0, 1, 1])

tbl = ax.table(cellText=df.values, colLabels=col_labels, cellLoc="center", loc="center",
    bbox=[0,0,1,1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.auto_set_column_width(col=list(range(n_cols)))

for j in range(n_cols):
    tbl[0, j].set_facecolor("#2c2c2c")
    tbl[0, j].set_text_props(color="white", fontweight="bold")

for i, (_, row) in enumerate(df.iterrows()):
    skill = row["Skill"]
    color = SKILL_COLORS[skill] if i % 2 == 0 else ROW_ALT[skill]
    for j in range(n_cols):
        tbl[i + 1, j].set_facecolor(color)
        if col_labels[j] == "XP in 30min":
            tbl[i + 1, j].set_text_props(fontweight="bold")

for i in range(1, n_rows + 1):
    tbl[i, 0].set_text_props(fontweight="bold")

ax.set_title(
    f"Endless Harvest — {MINUTES} min @ lvl {LEVEL}  |  ×2 resources  |  ×{XP_MULTIPLIER} XP multiplier\n"
    f"WC: rune axe ({WC_TICKS} ticks/roll)   Mining: rune pickaxe ({MINING_TICKS} ticks/roll)   Fishing: 4–6 ticks/roll (per spot)",
    fontsize=11, pad=8,
)

plt.savefig('endless_harvest.png', bbox_inches='tight')
plt.show()
print('Saved: endless_harvest.png')


In [ ]:
import pandas as pd

# ── Endless Harvest settings ──
XP_MULTIPLIER  = 16    # total XP multiplier — change as needed
MINUTES        = 30
TICKS          = int(MINUTES * 60 / 0.6)   # 3000 ticks

# Ticks per roll by skill/method
# Woodcutting: 4 ticks (universal, regardless of axe)
# Mining:      3 ticks (rune pickaxe — pickaxe tier affects speed, not success formula)
# Fishing:     5 ticks (standard for all methods)
WC_TICKS      = 4
MINING_TICKS  = 3

LEVEL = 99   # level assumed for all resources

rows = []

# ── Woodcutting (rune axe) ──
for tree, data in TREES.items():
    rolls   = TICKS / WC_TICKS
    xpa     = xp_per_action_wc(tree, LEVEL, axe="rune")
    xpa_x2  = xpa * 2
    rows.append({
        "Skill":          "Woodcutting",
        "Resource":       tree,
        "Ticks/Roll":     WC_TICKS,
        "Rolls/30min":    int(rolls),
        "XP/Action":      round(xpa, 2),
        "XP/Action (×2)": round(xpa_x2, 2),
        "XP in 30min":    round(rolls * xpa_x2 * XP_MULTIPLIER),
    })

# ── Mining (rune pickaxe) ──
plain_rocks = ["Copper", "Tin", "Iron", "Silver", "Coal", "Gold",
               "Mithril", "Adamantite", "Runite", "Amethyst",
               "Gem rock", "Gem rock (glory)"]
for rock in plain_rocks:
    rolls   = TICKS / MINING_TICKS
    xpa     = xp_per_action_mining(rock, LEVEL)
    xpa_x2  = xpa * 2
    rows.append({
        "Skill":          "Mining",
        "Resource":       rock,
        "Ticks/Roll":     MINING_TICKS,
        "Rolls/30min":    int(rolls),
        "XP/Action":      round(xpa, 2),
        "XP/Action (×2)": round(xpa_x2, 2),
        "XP in 30min":    round(rolls * xpa_x2 * XP_MULTIPLIER),
    })
for rock in ["Sandstone", "Granite"]:
    rolls   = TICKS / MINING_TICKS
    xpa     = xp_per_action_mining(rock, LEVEL)
    xpa_x2  = xpa * 2
    rows.append({
        "Skill":          "Mining",
        "Resource":       rock + " (priority roll)",
        "Ticks/Roll":     MINING_TICKS,
        "Rolls/30min":    int(rolls),
        "XP/Action":      round(xpa, 2),
        "XP/Action (×2)": round(xpa_x2, 2),
        "XP in 30min":    round(rolls * xpa_x2 * XP_MULTIPLIER),
    })

# ── Fishing (regular harpoon / default bait) ──
# Shared spots are listed once — xp_per_action_fishing already combines both fish
fishing_entries = [
    ("Shrimp/Anchovies",              "Shrimp",         dict()),
    ("Sardine/Herring",               "Sardine",        dict()),
    ("Pike",                          "Pike",           dict()),
    ("Trout/Salmon",                  "Trout",          dict()),
    ("Lobster",                       "Lobster",        dict()),
    ("Tuna/Swordfish",                "Tuna",           dict(harpoon="regular")),
    ("Bass",                          "Bass",           dict()),
    ("Monkfish",                      "Monkfish",       dict()),
    ("Karambwan",                     "Karambwan",      dict()),
    ("Shark",                         "Shark",          dict(harpoon="regular")),
    ("Anglerfish",         "Anglerfish",     dict(bait="sandworms")),
    ("Dark Crab",                     "Dark Crab",      dict()),
    ("Leaping Trout/Salmon/Sturgeon", "Leaping Trout",  dict()),
]
for label, fish, kwargs in fishing_entries:
    fticks  = FISHING_SPOT_TICKS[fish]
    rolls   = TICKS / fticks
    xpa     = xp_per_action_fishing(fish, LEVEL, **kwargs)
    xpa_x2  = xpa * 2
    rows.append({
        "Skill":          "Fishing",
        "Resource":       label,
        "Ticks/Roll":     fticks,
        "Rolls/30min":    int(rolls),
        "XP/Action":      round(xpa, 2),
        "XP/Action (×2)": round(xpa_x2, 2),
        "XP in 30min":    round(rolls * xpa_x2 * XP_MULTIPLIER),
    })

df = pd.DataFrame(rows)
df = df.sort_values(["Skill", "XP in 30min"], ascending=[True, False]).reset_index(drop=True)
df["XP in 30min"] = df["XP in 30min"].map("{:,.0f}".format)

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)
print(f"Endless Harvest — {MINUTES} min @ level {LEVEL}  |  ×2 resources  |  ×{XP_MULTIPLIER} XP multiplier")
print(f"Ticks/roll: WC={WC_TICKS} ({int(TICKS/WC_TICKS)} rolls)  Mining={MINING_TICKS} ({int(TICKS/MINING_TICKS)} rolls)  Fishing: 4-6 per spot\n")
print(df.to_string(index=False))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms

# ── Barbarian Gathering table settings ──
XP_MULTIPLIER      = 16
ITEMS_GATHERED     = 140 + 26
SUCCESSFUL_ACTIONS = ITEMS_GATHERED
LEVEL              = 99

WC_TICKS             = 4
CRYSTAL_MINING_TICKS = (3 * 3 + 2 * 1) / 4   # = 2.75 ticks avg

def _spot_p_total(fish, level, harpoon="regular", barbarian_gathering=False):
    """P(catch anything) for a fishing spot.  For shared spots with BG the
    action-level formula is used: BG fires once if the full chain fails."""
    spot      = FISHING_SPOTS[fish]
    reachable = [f for f in spot if level >= FISH[f]["level_req"]]
    is_shared = len(reachable) > 1

    prob_all_fail = 1.0
    for f in reachable:
        per_fish_bg = barbarian_gathering and not is_shared
        p = get_fish_chance(f, level, harpoon=harpoon, barbarian_gathering=per_fish_bg)
        prob_all_fail *= (1 - p)

    p_spot = 1 - prob_all_fail
    if barbarian_gathering and is_shared:
        p_spot *= (1 + 0.5 * prob_all_fail)
    return p_spot

def _sandgran_p_total(rock, level, barbarian_gathering=False):
    """P(get any piece) for Sandstone/Granite.  BG fires once if the full
    priority chain fails (action-level, not per-size)."""
    sizes = SANDSTONE_SIZES if rock == "Sandstone" else GRANITE_SIZES
    prob_all_fail = 1.0
    for size in sizes:
        p = get_mine_chance(size, level, barbarian_gathering=False)
        prob_all_fail *= (1 - p)
    p_spot = 1 - prob_all_fail
    if barbarian_gathering:
        p_spot *= (1 + 0.5 * prob_all_fail)
    return p_spot

def _fmt_time(seconds):
    m = int(seconds) // 60
    s = int(seconds) % 60
    return f"{m}m {s:02d}s"

rows = []

for tree, data in TREES.items():
    p   = get_cut_chance(tree, "dragon", LEVEL, crystal_axe=True, barbarian_gathering=True)
    xpa = xp_per_action_wc(tree, LEVEL, axe="dragon", crystal_axe=True, barbarian_gathering=True)
    exp_rolls = SUCCESSFUL_ACTIONS / p
    total_xp  = SUCCESSFUL_ACTIONS * (xpa / p) * XP_MULTIPLIER
    xph = xpa * (3600 / (WC_TICKS * 0.6)) * XP_MULTIPLIER
    rows.append({"Skill": "Woodcutting", "Resource": tree, "Crystal": "✓",
                 "p(success)": round(p, 4), "XP/Action": round(xpa, 2),
                 "XP/Hour": round(xph), "Total XP": round(total_xp), "Time": _fmt_time(exp_rolls * WC_TICKS * 0.6)})

for rock in ["Copper", "Tin", "Iron", "Silver", "Coal", "Gold",
             "Mithril", "Adamantite", "Runite", "Amethyst",
             "Gem rock (glory)"]:
    p   = get_mine_chance(rock, LEVEL, barbarian_gathering=True)
    xpa = xp_per_action_mining(rock, LEVEL, barbarian_gathering=True)
    exp_rolls = SUCCESSFUL_ACTIONS / p
    total_xp  = SUCCESSFUL_ACTIONS * (xpa / p) * XP_MULTIPLIER
    xph = xpa * (3600 / (CRYSTAL_MINING_TICKS * 0.6)) * XP_MULTIPLIER
    rows.append({"Skill": "Mining", "Resource": rock, "Crystal": "✓",
                 "p(success)": round(p, 4), "XP/Action": round(xpa, 2),
                 "XP/Hour": round(xph), "Total XP": round(total_xp), "Time": _fmt_time(exp_rolls * CRYSTAL_MINING_TICKS * 0.6)})

for rock in ["Sandstone", "Granite"]:
    p   = _sandgran_p_total(rock, LEVEL, barbarian_gathering=True)
    xpa = xp_per_action_mining(rock, LEVEL, barbarian_gathering=True)
    exp_rolls = SUCCESSFUL_ACTIONS / p
    total_xp  = SUCCESSFUL_ACTIONS * (xpa / p) * XP_MULTIPLIER
    xph = xpa * (3600 / (CRYSTAL_MINING_TICKS * 0.6)) * XP_MULTIPLIER
    rows.append({"Skill": "Mining", "Resource": rock + " (priority roll)", "Crystal": "✓",
                 "p(success)": round(p, 4), "XP/Action": round(xpa, 2),
                 "XP/Hour": round(xph), "Total XP": round(total_xp), "Time": _fmt_time(exp_rolls * CRYSTAL_MINING_TICKS * 0.6)})

fishing_entries = [
    ("Shrimp/Anchovies",              "Shrimp",        dict(harpoon="regular")),
    ("Sardine/Herring",               "Sardine",       dict(harpoon="regular")),
    ("Pike",                          "Pike",          dict(harpoon="regular")),
    ("Trout/Salmon",                  "Trout",         dict(harpoon="regular")),
    ("Lobster",                       "Lobster",       dict(harpoon="regular")),
    ("Tuna/Swordfish",                "Tuna",          dict(harpoon="crystal")),
    ("Bass",                          "Bass",          dict(harpoon="regular")),
    ("Monkfish",                      "Monkfish",      dict(harpoon="regular")),
    ("Karambwan",                     "Karambwan",     dict(harpoon="regular")),
    ("Shark",                         "Shark",         dict(harpoon="crystal")),
    ("Anglerfish",                    "Anglerfish",    dict(bait="sandworms",      harpoon="regular")),
    ("Dark Crab",                     "Dark Crab",     dict(harpoon="regular")),
    ("Leaping Trout/Salmon/Sturgeon", "Leaping Trout", dict(harpoon="regular")),
]
for label, fish, kwargs in fishing_entries:
    harpoon = kwargs.get("harpoon", "regular")
    bait    = kwargs.get("bait", "sandworms")
    p   = _spot_p_total(fish, LEVEL, harpoon=harpoon, barbarian_gathering=True)
    xpa = xp_per_action_fishing(fish, LEVEL, harpoon=harpoon, bait=bait, barbarian_gathering=True)
    exp_rolls = SUCCESSFUL_ACTIONS / p
    total_xp  = SUCCESSFUL_ACTIONS * (xpa / p) * XP_MULTIPLIER
    fticks = FISHING_SPOT_TICKS[fish]
    xph = xpa * (3600 / (fticks * 0.6)) * XP_MULTIPLIER
    rows.append({"Skill": "Fishing", "Resource": label,
                 "Crystal": "✓" if harpoon == "crystal" else "✗",
                 "p(success)": round(p, 4), "XP/Action": round(xpa, 2),
                 "XP/Hour": round(xph), "Total XP": round(total_xp), "Time": _fmt_time(exp_rolls * fticks * 0.6)})

df_bg = pd.DataFrame(rows)
df_bg = df_bg.sort_values(["Skill", "XP/Hour"], ascending=[True, False]).reset_index(drop=True)
df_bg["XP/Hour"] = df_bg["XP/Hour"].map("{:,.0f}".format)
df_bg["Total XP"] = df_bg["Total XP"].map("{:,.0f}".format)

SKILL_COLORS = {"Woodcutting": "#d4f0c0", "Mining": "#d0e8f5", "Fishing": "#fde9c0"}
ROW_ALT      = {"Woodcutting": "#eaf7e0", "Mining": "#e5f2fa", "Fishing": "#fef4db"}

col_labels = list(df_bg.columns)
n_rows, n_cols = df_bg.shape

fig, ax = plt.subplots(figsize=(14, 0.45 * n_rows + 1.2))
ax.axis("off")
ax.set_position([0, 0, 1, 1])

tbl = ax.table(cellText=df_bg.values, colLabels=col_labels, cellLoc="center", loc="center",
    bbox=[0,0,1,1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.auto_set_column_width(col=list(range(n_cols)))

for j in range(n_cols):
    tbl[0, j].set_facecolor("#2c2c2c")
    tbl[0, j].set_text_props(color="white", fontweight="bold")

for i, (_, row) in enumerate(df_bg.iterrows()):
    skill = row["Skill"]
    color = SKILL_COLORS[skill] if i % 2 == 0 else ROW_ALT[skill]
    for j in range(n_cols):
        tbl[i + 1, j].set_facecolor(color)
        if col_labels[j] in ("XP/Hour", "Total XP", "Time"):
            tbl[i + 1, j].set_text_props(fontweight="bold")

for i in range(1, n_rows + 1):
    tbl[i, 0].set_text_props(fontweight="bold")

ax.set_title(
    f"Barbarian Gathering — crystal tools  |  BG active  |  ×{XP_MULTIPLIER} XP\n"
    f"Mining: crystal pickaxe ({CRYSTAL_MINING_TICKS:.2f} ticks avg)  |  "
    f"Stop at {ITEMS_GATHERED} items  |  lvl {LEVEL}",
    fontsize=11, pad=8,
)

plt.savefig('barbarian_gathering.png', bbox_inches='tight')
plt.show()
print('Saved: barbarian_gathering.png')
